## Chains

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df_o = pd.read_excel('adriel_qualitativas.xlsx')

df = df_o[:]

df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace('\n', '', regex=True)

print(len(df))

print(df.columns.tolist())

95
['ID', 'Carimbo de data/hora', 'Após ter recebido esclarecimentos sobre a natureza da pesquisa, seus objetivos e métodos, eu declaro que sou maior de 18 anos e aceito participar por meio do formulário de participação, conforme as disposições da Lei Geral de Proteção de Dados (Lei nº 13.709/2018).Em caso de resposta negativa, o formulário será encerrado, garantindo assim o respeito à autonomia e privacidade dos participantes, conforme estabelecido pela legislação vigente.', 'Qual é a sua idade?', 'Em qual estado do Brasil você mora atualmente?', 'Qual é o seu gênero?', 'Qual é o seu nível de educação formal concluída?', 'Qual seu modelo de trabalho?', 'Qual papel que melhor descreve suas atividades atuais em projetos de desenvolvimento de software?', 'Por favor, indique a senioridade da posição que ocupa.', 'Quantos anos de experiência você tem em funções relacionadas ao desenvolvimento de software ou TI?', 'Quantos anos de experiência você tem em funções relacionadas a Segurança de 

In [4]:
print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())


df = df[~(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?']
          == 'Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software')]

print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())
print(df.shape)

['Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software'
 'Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.'
 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
['Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.'
 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
(83, 103)


In [5]:
df.isna().sum()

ID                                                                                                                                                                                                                                                                                                                                                                                                                                                              0
Carimbo de data/hora                                                                                                                                                                                                                                                                                                                                                                                                                                            0
Após ter recebido esclarecimentos sobre a natureza da pesquisa, seus objetivos e métodos, eu declaro

In [6]:
df.shape

(83, 103)

Analises

In [7]:
categorias = {
    "3rd Party Vulnerability Scanning": [
        "Dependabot", "PHP Security Checker", "Snyk", "Clair", "Blackduck",
        "Retire.JS", "SRC:CLR", "Anchore", "Node Security Platform", "Trivy",
        "Sonatype", "Whitesource"
    ],
    "Static Application Security Testing": [  # SAST
        "CodeQL", "CheckStyle", "Coverity", "SonarQube", "Bandit", "PHPStan",
        "LGTM", "Brakeman", "ESLint Security", "Dart Analyzer", "DevSkim"
    ],
    "Dynamic Application Security Testing": [  # DAST
        "Qark", "Checkmarx", "CFN NAG"
    ],
    "Secure Configuration/Hardening": [
        "Chef", "Ansible", "Terraform", "Puppet", "Phan", "Seccomp",
        "CloudCustodian", "Turbot"
    ],
    "Compliance/Hardening Checks": [
        "Bane", "Ansible-Lint", "Blackbox", "Puppet-Lint", "InSpec",
        "Kube Audit", "Foodcritic", "Cookstyle", "conftest"
    ],
    "Secrets Management": [
        "Pinterest Knox", "GitLeaks", "CyberArk Conjur",
        "ThoughtWorks Talisman", "Berglas", "Docker Secrets", "Trufflehog"
    ]
}

In [8]:
cols_ferramentas = [
    col for col in df.columns
    if "categorias de ferramentas" in str(col).lower()
]

print(f"\n{len(cols_ferramentas)} colunas encontradas para a pergunta das ferramentas:")
for c in cols_ferramentas:
    print("-", c)

df_ferramentas = df[cols_ferramentas].copy()


6 colunas encontradas para a pergunta das ferramentas:
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [3rd Party Vulnerability Scanning Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Static Application Security Testing Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Dynamic Application Security Testing Tools]
- Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere mar

In [9]:
df_ferramentas = df_ferramentas.applymap(
    lambda x: x.strip() if isinstance(x, str) else x
)

C:\Users\RESET Lab\AppData\Local\Temp\ipykernel_12676\389216647.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_ferramentas = df_ferramentas.applymap(


In [10]:
df_ferramentas.insert(0, "Respondente", range(1, len(df_ferramentas) + 1))

In [11]:
escala_likert = [
    "Diariamente",
    "Semanalmente",
    "Mensalmente",
    "Anualmente",
    "Não Aplicável",
]

In [12]:
resultado = {}

for categoria in df_ferramentas.columns[1:]:  # ignora a coluna "Respondente"
    resultado[categoria] = {}
    for opcao in escala_likert:
        # Seleciona todas as linhas onde o respondente marcou essa opção
        participantes = df_ferramentas[df_ferramentas[categoria] == opcao][[
            "Respondente", categoria]]

        # Armazena as respostas individuais
        resultado[categoria][opcao] = participantes

In [13]:
exemplo_categoria = list(resultado.keys())[2]
print(f"\nExemplo — Categoria: {exemplo_categoria}")
for opcao, respostas in resultado[exemplo_categoria].items():
    print(f"\nOpção: {opcao}")
    print(respostas)


Exemplo — Categoria: Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Dynamic Application Security Testing Tools]

Opção: Diariamente
    Respondente  \
4             4   
5             5   
24           21   
28           24   
32           28   
33           29   
34           30   
37           33   
54           47   
63           56   
75           66   
77           68   
80           71   
84           75   
91           81   
92           82   

   Para cada uma das seguintes categorias de ferramentas de segurança automatizadas, indique com que frequência você as utiliza.  Considere marcar “Não Aplicável” para as categorias de ferramentas das quais você não utiliza. [Dynamic Application Security Testing Tools]  
4                                         Diariamente                                          

In [14]:
with pd.ExcelWriter("respostas_individuais_agrupadas.xlsx") as writer:
    for i, (categoria, respostas_por_opcao) in enumerate(resultado.items()):
        frames = []
        for opcao, respostas in respostas_por_opcao.items():
            temp = respostas.copy()
            temp["Opção"] = opcao
            frames.append(temp)
        df_categoria = pd.concat(frames, ignore_index=True)
        # garante unicidade e respeita limite de 31 chars
        nome_aba = f"{categoria[:27]}_{i}"
        df_categoria.to_excel(writer, sheet_name=nome_aba, index=False)

print("\nArquivo salvo: respostas_individuais_agrupadas.xlsx")


Arquivo salvo: respostas_individuais_agrupadas.xlsx


## Não aplicavel


In [15]:
# Lista de colunas de preocupação
colunas_preocupacao = [
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Revisão de Código]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Modelagem de Ameaças]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Identificação de Casos de Abuso/Mau Uso]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Análise de Riscos Arquiteturais]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Penetração]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Riscos]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Vulnerabilidades]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Segurança Manuais/Automatizados]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Execução de Testes Fuzz]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Engenharia de Requisitos de Segurança]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Codificação Segura]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Resposta Emergencial ao Código]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Configurações e Gerenciamento de Segurança]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Monitoramento de Eventos de Segurança]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Auditorias de Segurança Externas]',
    'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Segurança da Cadeia de Suprimentos de Software]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Revisão de Código]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Modelagem de Ameaças]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Identificação de Casos de Abuso/Mau Uso]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Análise de Riscos Arquiteturais]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Penetração]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Riscos]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Vulnerabilidades]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Segurança Manuais/Automatizados]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Execução de Testes Fuzz]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Engenharia de Requisitos de Segurança]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Codificação Segura]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Resposta Emergencial ao Código]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Configurações e Gerenciamento de Segurança]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Monitoramento de Eventos de Segurança]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Auditorias de Segurança Externas]',
    'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Segurança da Cadeia de Suprimentos de Software]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Revisão de Código]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Modelagem de Ameaças]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Identificação de Casos de Abuso/Mau Uso]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Análise de Riscos Arquiteturais]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Penetração]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Riscos]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Vulnerabilidades]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Segurança Manuais/Automatizados]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Execução de Testes Fuzz]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Engenharia de Requisitos de Segurança]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Codificação Segura]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Resposta Emergencial ao Código]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Configurações e Gerenciamento de Segurança]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Monitoramento de Eventos de Segurança]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Auditorias de Segurança Externas]',
    'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Segurança da Cadeia de Suprimentos de Software]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Revisão de Código]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Modelagem de Ameaças]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Identificação de Casos de Abuso/Mau Uso]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Análise de Riscos Arquiteturais]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Testes de Penetração]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Gerenciamento de Riscos]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Gerenciamento de Vulnerabilidades]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Testes de Segurança Manuais/Automatizados]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Execução de Testes Fuzz]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Engenharia de Requisitos de Segurança]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Codificação Segura]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Resposta Emergencial ao Código]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Configurações e Gerenciamento de Segurança]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Monitoramento de Eventos de Segurança]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Auditorias de Segurança Externas]',
    'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Segurança da Cadeia de Suprimentos de Software]'
]

In [16]:
nao_aplicavel_summary = []

for coluna in colunas_preocupacao:
    # Contar as ocorrências de 'Não Aplicável'
    nao_aplicavel_count = df[coluna].value_counts().get('Não Aplicável', 0)

    # Contar o total de respostas válidas (excluindo NaN para o cálculo da porcentagem do que foi respondido)
    total_respostas_validas = df[coluna].dropna().shape[0]

    if total_respostas_validas > 0:
        nao_aplicavel_percentage = (
            nao_aplicavel_count / total_respostas_validas) * 100
    else:
        nao_aplicavel_percentage = 0

    nao_aplicavel_summary.append({
        'Coluna': coluna,
        'Contagem Não Aplicável': nao_aplicavel_count,
        'Porcentagem Não Aplicável': round(nao_aplicavel_percentage, 2)
    })

df_nao_aplicavel_summary = pd.DataFrame(nao_aplicavel_summary)

print("DataFrame com o resumo de 'Não Aplicável' por coluna:")
display(df_nao_aplicavel_summary)

DataFrame com o resumo de 'Não Aplicável' por coluna:


,Coluna,Contagem Não Aplicável,Porcentagem Não Aplicável
0,"Com base em sua experiência, como você avaliar...",11,13.25
1,"Com base em sua experiência, como você avaliar...",20,24.10
2,"Com base em sua experiência, como você avaliar...",18,21.69
3,"Com base em sua experiência, como você avaliar...",14,16.87
4,"Com base em sua experiência, como você avaliar...",15,18.07
...,...,...,...
59,Qual a sua opinião quanto a frase a seguir apl...,0,0.00
60,Qual a sua opinião quanto a frase a seguir apl...,0,0.00
61,Qual a sua opinião quanto a frase a seguir apl...,0,0.00
62,Qual a sua opinião quanto a frase a seguir apl...,0,0.00


In [17]:
# Identificar colunas de 'facilidade' e 'efetividade'
colunas_facilidade = [col for col in df_nao_aplicavel_summary['Coluna']
                      if 'facilidade de uso' in col.lower()]
colunas_efetividade = [col for col in df_nao_aplicavel_summary['Coluna']
                       if 'efetividade das seguintes práticas' in col.lower()]

# Filtrar o DataFrame de resumo para obter as porcentagens
df_facilidade = df_nao_aplicavel_summary[df_nao_aplicavel_summary['Coluna'].isin(
    colunas_facilidade)]
df_efetividade = df_nao_aplicavel_summary[df_nao_aplicavel_summary['Coluna'].isin(
    colunas_efetividade)]

# Calcular a média das porcentagens para cada grupo
media_facilidade = df_facilidade['Porcentagem Não Aplicável'].mean()
media_efetividade = df_efetividade['Porcentagem Não Aplicável'].mean()

# Criar um novo DataFrame com os resultados
df_resumo_aplicavel = pd.DataFrame({
    'Atributo': ['Facilidade de Uso', 'Efetividade'],
    'Média Porcentagem Não Aplicável': [round(media_facilidade, 2), round(media_efetividade, 2)]
})

print("DataFrame com o resumo da média de 'Não Aplicável' por atributo:")
display(df_resumo_aplicavel)

DataFrame com o resumo da média de 'Não Aplicável' por atributo:


,Atributo,Média Porcentagem Não Aplicável
0,Facilidade de Uso,22.74
1,Efetividade,19.35


## normalização

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# pd.set_option('display.max_rows', None)

In [19]:
df_o = pd.read_excel(
    'Cópia 2 de [PT-BR] Script do Questionário (respostas) (1).xlsx')

df = df_o[:]

df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace('\n', '', regex=True)

print(len(df))

print(df.columns.tolist())

95
['Carimbo de data/hora', 'Após ter recebido esclarecimentos sobre a natureza da pesquisa, seus objetivos e métodos, eu declaro que sou maior de 18 anos e aceito participar por meio do formulário de participação, conforme as disposições da Lei Geral de Proteção de Dados (Lei nº 13.709/2018).Em caso de resposta negativa, o formulário será encerrado, garantindo assim o respeito à autonomia e privacidade dos participantes, conforme estabelecido pela legislação vigente.', 'Qual é a sua idade?', 'Em qual estado do Brasil você mora atualmente?', 'Qual é o seu gênero?', 'Qual é o seu nível de educação formal concluída?', 'Qual seu modelo de trabalho?', 'Qual papel que melhor descreve suas atividades atuais em projetos de desenvolvimento de software?', 'Por favor, indique a senioridade da posição que ocupa.', 'Quantos anos de experiência você tem em funções relacionadas ao desenvolvimento de software ou TI?', 'Quantos anos de experiência você tem em funções relacionadas a Segurança de Softwa

In [20]:
print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())


df = df[~(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?']
          == 'Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software')]

print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].unique())
print(df.shape)

['Nunca tive experiência direta e nem indireta com segurança no processo de desenvolvimento de software'
 'Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.'
 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
['Meu grupo/equipe de desenvolvimento trabalha (ou trabalhou) com segurança, mas não estou diretamente envolvido em nenhuma tarefa.'
 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.']
(83, 102)


In [21]:
df[df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].isna()]

,Carimbo de data/hora,"Após ter recebido esclarecimentos sobre a natureza da pesquisa, seus objetivos e métodos, eu declaro que sou maior de 18 anos e aceito participar por meio do formulário de participação, conforme as disposições da Lei Geral de Proteção de Dados (Lei nº 13.709/2018).Em caso de resposta negativa, o formulário será encerrado, garantindo assim o respeito à autonomia e privacidade dos participantes, conforme estabelecido pela legislação vigente.",Qual é a sua idade?,Em qual estado do Brasil você mora atualmente?,Qual é o seu gênero?,Qual é o seu nível de educação formal concluída?,Qual seu modelo de trabalho?,Qual papel que melhor descreve suas atividades atuais em projetos de desenvolvimento de software?,"Por favor, indique a senioridade da posição que ocupa.",Quantos anos de experiência você tem em funções relacionadas ao desenvolvimento de software ou TI?,...,Em quais fases do ciclo de desenvolvimento de software você costuma utilizar as ferramentas de segurança automatizadas selecionadas na pergunta anterior? (Selecione todas as opções que se aplicam),Quais recomendações você daria para um desenvolvedor sobre a adoção de ferramentas de segurança para seus projetos?,Quais são os principais desafios que você enfrenta ao automatizar ferramentas de segurança? (Selecione todas as opções que se aplicam),Quais melhorias você gostaria de ver nas ferramentas de segurança automatizadas que utiliza atualmente? (Selecione todas as opções que se aplicam),"Caso você queira receber os resultados desse estudo e participar de uma futura entrevista sobre Práticas de Segurança de Software, por favor, informe seu Email.","Quais são as razões pelas quais você opta por não utilizar ferramentas automatizadas de segurança de software? Por favor, descreva brevemente sobre.",Você considera utilizar ferramentas automatizadas de segurança de software em seus projetos no futuro?,"Se você respondeu ""Sim"" ou ""Talvez"" à pergunta anterior, por favor, indique quais ferramentas utilizaria. Marque ""Nenhuma"" se respondeu ""Não"".","Caso você queira receber os resultados desse estudo e participar de uma futura entrevista sobre Práticas de Segurança de Software, por favor, informe seu Email..1",Marque a imagem em que aparece um filhote de cachorro:


In [22]:
print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].isna().sum())

df = df.drop(df[df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].isna()].index, axis=0)

print(df['Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'].isna().sum())

0
0


Peso

In [23]:
praticas = ['[Revisão de Código]',
            '[Modelagem de Ameaças]',
            '[Identificação de Casos de Abuso/Mau Uso]',
            '[Análise de Riscos Arquiteturais]',
            '[Testes de Penetração]',
            '[Gerenciamento de Riscos]',
            '[Gerenciamento de Vulnerabilidades]',
            '[Testes de Segurança Manuais/Automatizados]',
            '[Execução de Testes Fuzz]',
            '[Engenharia de Requisitos de Segurança]',
            '[Codificação Segura]',
            '[Resposta Emergencial ao Código]',
            '[Configurações e Gerenciamento de Segurança]',
            '[Monitoramento de Eventos de Segurança]',
            '[Auditorias de Segurança Externas]',
            '[Segurança da Cadeia de Suprimentos de Software]']

In [24]:
colunas_frequencia = ['Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Revisão de Código]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Modelagem de Ameaças]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Identificação de Casos de Abuso/Mau Uso]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Análise de Riscos Arquiteturais]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Penetração]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Riscos]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Vulnerabilidades]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Segurança Manuais/Automatizados]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Execução de Testes Fuzz]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Engenharia de Requisitos de Segurança]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Codificação Segura]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Resposta Emergencial ao Código]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Configurações e Gerenciamento de Segurança]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Monitoramento de Eventos de Segurança]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Auditorias de Segurança Externas]',
                      'Com que frequência você costuma aplicar as seguintes práticas de segurança de software  em seus projetos? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Segurança da Cadeia de Suprimentos de Software]'
                      ]

In [25]:
colunas_facilidade = ['Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Revisão de Código]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Modelagem de Ameaças]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Identificação de Casos de Abuso/Mau Uso]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Análise de Riscos Arquiteturais]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Penetração]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Riscos]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Gerenciamento de Vulnerabilidades]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Testes de Segurança Manuais/Automatizados]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Execução de Testes Fuzz]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Engenharia de Requisitos de Segurança]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Codificação Segura]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Resposta Emergencial ao Código]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Configurações e Gerenciamento de Segurança]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Monitoramento de Eventos de Segurança]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Auditorias de Segurança Externas]',
                      'Com base em sua experiência, como você avaliaria o nível de facilidade de uso das seguintes práticas de segurança de software? Considere marcar “Não Aplicável” para as práticas das quais você não utiliza.  [Segurança da Cadeia de Suprimentos de Software]'
                      ]

In [26]:
colunas_efetividade = ['Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Revisão de Código]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Modelagem de Ameaças]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Identificação de Casos de Abuso/Mau Uso]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Análise de Riscos Arquiteturais]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Penetração]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Riscos]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Gerenciamento de Vulnerabilidades]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Testes de Segurança Manuais/Automatizados]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Execução de Testes Fuzz]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Engenharia de Requisitos de Segurança]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Codificação Segura]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Resposta Emergencial ao Código]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Configurações e Gerenciamento de Segurança]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Monitoramento de Eventos de Segurança]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Auditorias de Segurança Externas]',
                       'Com base em sua experiência, como você avalia a efetividade das seguintes práticas de segurança de software na prevenção, remoção ou detecção de problemas de segurança em seus projetos?  Considere marcar “Não Aplicável” para as práticas das quais você não utiliza. [Segurança da Cadeia de Suprimentos de Software]'
                       ]

In [27]:
colunas_essencialidade = ['Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Revisão de Código]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Modelagem de Ameaças]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Identificação de Casos de Abuso/Mau Uso]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Análise de Riscos Arquiteturais]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Testes de Penetração]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Gerenciamento de Riscos]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Gerenciamento de Vulnerabilidades]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Testes de Segurança Manuais/Automatizados]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Execução de Testes Fuzz]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Engenharia de Requisitos de Segurança]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Codificação Segura]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Resposta Emergencial ao Código]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Configurações e Gerenciamento de Segurança]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Monitoramento de Eventos de Segurança]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Auditorias de Segurança Externas]',
                          'Qual a sua opinião quanto a frase a seguir aplicada em cada uma das práticas abaixo:“Essa prática é essencial para a proteção do sistema.” [Segurança da Cadeia de Suprimentos de Software]']

In [28]:
frequencia_map = {
    'Não aplicável': 0,
    'Anualmente': 1,
    'Mensalmente': 2,
    'Semanalmente': 3,
    'Diariamente': 4,
}

efetividade_map = {
    'Não Aplicável': 0,
    'Muito Baixo': 1,
    'Baixo': 2,
    'Moderado': 3,
    'Alto': 4,
    'Muito Alto': 5,
}

facilidade_map = {
    'Não Aplicável': 0,
    'Muito Baixo': 1,
    'Baixo': 2,
    'Moderado': 3,
    'Alto': 4,
    'Muito Alto': 5,
}

essencialidade_map = {
    'Discordo Totalmente': 1,
    'Discordo': 2,
    'Neutro': 3,
    'Concordo': 4,
    'Concordo Totalmente': 5,
}

In [29]:
def plot_barras(
    df_freq: pd.DataFrame,
    titulo: str = "Frequência das Práticas de Segurança",
    categorias: list = None
):
    """
    Cria um gráfico de barras horizontais empilhadas com as categorias de frequência.
    Detecta automaticamente as categorias presentes no DataFrame.
    """

    # Se categorias não forem passadas, usar as colunas do df
    if categorias is None:
        categorias = list(df_freq.columns)

    # Converter contagens em porcentagens
    df_pct = df_freq.div(df_freq.sum(axis=1), axis=0) * 100

    # Definir paleta adaptável (até 6 cores suaves)
    paleta_padrao = [
        "#cfcfcf", "#d4b483", "#b3cde3", "#8c96c6", "#5e4fa2", "#542788"
    ]
    cores = {cat: paleta_padrao[i % len(paleta_padrao)]
             for i, cat in enumerate(categorias)}

    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    bottom = pd.Series([0] * len(df_pct), index=df_pct.index)

    for cat in categorias:
        if cat in df_pct.columns:
            ax.barh(df_pct.index, df_pct[cat],
                    left=bottom, color=cores[cat], label=cat)
            bottom += df_pct[cat]

    # Estilo
    ax.set_xlabel("Percentual (%)")
    ax.set_ylabel("Atividade de Segurança")
    ax.set_title(titulo, fontsize=13, fontweight='bold', pad=15)
    ax.legend(
        title="Frequência",
        bbox_to_anchor=(1.05, 1),
        loc='upper left',
        frameon=False
    )
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    plt.show()

In [30]:
def gerar_tabela_descricao(df: pd.DataFrame, ordem: list = None) -> pd.DataFrame:
    """
    Gera uma tabela de contagem de frequências (Não aplicável, Anualmente, etc.)
    para cada atividade de segurança contida no DataFrame.

    Parâmetros:
        df (pd.DataFrame): DataFrame com colunas de atividades e valores como
                           'Diariamente', 'Semanalmente', 'Mensalmente', etc.

    Retorna:
        pd.DataFrame: Tabela agregada com contagens por frequência.
    """
    # Copiar para evitar alterar o original
    desc_df = df.copy()

    # Ajustar nomes das colunas se estiverem no formato "coluna [x]"
    desc_df.columns = [c.split("[")[1].replace("]", "")
                       if "[" in c else c for c in desc_df.columns]
    desc_df = desc_df.rename(
        {"Revisão de Código": "Práticas de Segurança"}, axis=1)

    # Criar tabela de contagens
    descricao_frequencia = pd.DataFrame()
    for col in desc_df.columns:
        counts = desc_df[col].value_counts()
        descricao_frequencia[col] = counts

    # Organizar formato final (atividades nas linhas)
    descricao_frequencia = descricao_frequencia.fillna(0).astype(int).T

    # Reordenar colunas principais
    descricao_frequencia = descricao_frequencia.reindex(
        columns=ordem, fill_value=0)

    return descricao_frequencia


descricao_frequencia = gerar_tabela_descricao(
    df[colunas_frequencia], ordem=frequencia_map.keys())
descricao_facilidade = gerar_tabela_descricao(
    df[colunas_facilidade], ordem=facilidade_map.keys())
descricao_efetividade = gerar_tabela_descricao(
    df[colunas_efetividade], ordem=efetividade_map.keys())

In [31]:
df[colunas_essencialidade[0]].unique()

array(['Concordo', 'Concordo Totalmente', 'Neutro', 'Discordo Totalmente'],
      dtype=object)

In [32]:
for coluna in colunas_frequencia:
    df[coluna] = df[coluna].map(frequencia_map)

for coluna in colunas_efetividade:
    df[coluna] = df[coluna].map(efetividade_map)

for coluna in colunas_facilidade:
    df[coluna] = df[coluna].map(facilidade_map)

for coluna in colunas_essencialidade:
    df[coluna] = df[coluna].map(essencialidade_map)

In [33]:
df['Quantos anos de experiência você tem em funções relacionadas ao desenvolvimento de software ou TI?'].unique()

array(['Entre 7 e 14 anos', 'Entre 4 e 5 anos', 'Entre 1 e 3 anos',
       'Menos de 1 ano', 'Mais de 15 anos'], dtype=object)

In [34]:
coluna_experiencia = 'Você tem (ou teve) alguma experiência de trabalho relacionada à segurança de software no processo de desenvolvimento de software?'
coluna_anos_experiencia_dev = 'Quantos anos de experiência você tem em funções relacionadas ao desenvolvimento de software ou TI?'
coluna_anos_experiencia_seg = 'Quantos anos de experiência você tem em funções relacionadas a Segurança de Software?'
coluna_nivel_educacao = 'Qual é o seu nível de educação formal concluída?'
coluna_senioridade = 'Por favor, indique a senioridade da posição que ocupa.'
coluna_especialistas = 'Qual papel que melhor descreve suas atividades atuais em projetos de desenvolvimento de software?'
coluna_certificacao = 'Você possui alguma certificação na área de segurança da informação? Se sim, quais são elas?'

tem_experiencia = 'Eu já trabalhei (ou trabalho) com segurança no processo de desenvolvimento de software.'

resposta_especialistas = ["Cryptographer", "Penetration tester",
                          "DevOps engineer", "Privacy engineer", "Security engineer"]
resposta_nao_certificacao = ["Não possui nenhuma certificação"]

In [35]:
print(df['Qual é o seu nível de educação formal concluída?'].unique())
print(df[coluna_senioridade].unique())
print(df[coluna_anos_experiencia_dev].unique())
print(df[coluna_anos_experiencia_seg].unique())

['Mestrado' 'Estudante de Doutorado' 'Ensino Superior' 'Doutorado'
 'Especialização' 'Ensino Médio' 'Estudante de Mestrado']
['Sênior (10+ anos)' 'Júnior (até 5 anos)' 'Pleno (6 a 9 anos)'
 'Estagiário']
['Entre 7 e 14 anos' 'Entre 4 e 5 anos' 'Entre 1 e 3 anos'
 'Menos de 1 ano' 'Mais de 15 anos']
['Entre 7 e 14 anos' 'Nenhuma' 'Entre 1 e 3 anos' 'Entre 4 e 6 anos'
 'Menos de 1 ano' 'Mais de 15 anos']


In [36]:
len(df[~df[coluna_especialistas].isin(resposta_especialistas)]
    [coluna_especialistas].unique())

39

In [37]:
df[~df[coluna_especialistas].isin(
    resposta_especialistas)][coluna_especialistas].unique()

array(['Penetration tester, Security engineer',
       'Developer (Backend, frontend, fullstack)',
       'Penetration tester, Requirements Engineer, Security engineer',
       'DevOps engineer, Solution Architect',
       'Privacy engineer, Security engineer',
       'DevOps engineer, Privacy engineer, Security engineer',
       'Solution Architect, Test Manager / Tester', 'SOC',
       'Gerente de Segurança da Informação', 'Application secure',
       'Security Analyst',
       'Developer (Backend, frontend, fullstack), DevOps engineer, Security engineer, Solution Architect',
       'Appsec', 'Penetration tester, DevOps engineer, Security engineer',
       'Penetration tester, DevOps engineer, Security engineer, Solution Architect',
       'Penetration tester, DevOps engineer', 'Solution Architect',
       'Cybersegurança',
       'Data Scientist, Developer (Backend, frontend, fullstack), Penetration tester, DevOps engineer, Product owner, Project Lead / Project Manager, Security eng

In [38]:
print("Quantidade de especialistas:",
      df[df[coluna_especialistas].isin(resposta_especialistas)].shape[0])
print("Quantidade de não especialistas:",
      df[~df[coluna_especialistas].isin(resposta_especialistas)].shape[0])

Quantidade de especialistas: 26
Quantidade de não especialistas: 57


In [39]:
df_exp_direta = df[df[coluna_experiencia] == tem_experiencia]
df_exp_indireta = df[df[coluna_experiencia] != tem_experiencia]

df_exp_direta.shape

(62, 102)

In [40]:
df[coluna_senioridade].unique()

array(['Sênior (10+ anos)', 'Júnior (até 5 anos)', 'Pleno (6 a 9 anos)',
       'Estagiário'], dtype=object)

In [41]:
print(df['Qual é o seu nível de educação formal concluída?'].unique())
print(df[coluna_senioridade].unique())
print(df[coluna_anos_experiencia_dev].unique())
print(df[coluna_anos_experiencia_seg].unique())

['Mestrado' 'Estudante de Doutorado' 'Ensino Superior' 'Doutorado'
 'Especialização' 'Ensino Médio' 'Estudante de Mestrado']
['Sênior (10+ anos)' 'Júnior (até 5 anos)' 'Pleno (6 a 9 anos)'
 'Estagiário']
['Entre 7 e 14 anos' 'Entre 4 e 5 anos' 'Entre 1 e 3 anos'
 'Menos de 1 ano' 'Mais de 15 anos']
['Entre 7 e 14 anos' 'Nenhuma' 'Entre 1 e 3 anos' 'Entre 4 e 6 anos'
 'Menos de 1 ano' 'Mais de 15 anos']


In [42]:
nivel_educacao_map = {
    'Ensino Médio': 1,
    'Ensino Superior': 2,
    'Especialização': 3,
    'Estudante de Mestrado': 4,
    'Mestrado': 5,
    'Estudante de Doutorado': 6,
    'Doutorado': 7
}

senioridade_map = {
    'Estagiário': 1,
    'Júnior (até 5 anos)': 2,
    'Pleno (6 a 9 anos)': 3,
    'Sênior (10+ anos)': 4
}

anos_exp_dev_map = {
    'Menos de 1 ano': 1,
    'Entre 1 e 3 anos': 2,
    'Entre 4 e 5 anos': 3,
    'Entre 7 e 14 anos': 4,
    'Mais de 15 anos': 5
}

anos_exp_seg_map = {
    'Nenhuma': 0,
    'Menos de 1 ano': 1,
    'Entre 1 e 3 anos': 2,
    'Entre 4 e 6 anos': 3,
    'Entre 7 e 14 anos': 4,
    'Mais de 15 anos': 5
}


def interacao_exp_direta_especialista(row):
    if row['especialista'] and row['experiencia_direta']:
        return 3
    elif row['experiencia_direta'] and not row['especialista']:
        return 2
    else:
        return 1


def relacao_exp_direta_especialista(row):
    return (row['especialista'] + row['experiencia_direta'])/2


df[coluna_anos_experiencia_dev] = df[coluna_anos_experiencia_dev].map(
    anos_exp_dev_map)

df[coluna_senioridade] = df[coluna_senioridade].map(senioridade_map)

df[coluna_anos_experiencia_seg] = df[coluna_anos_experiencia_seg].map(
    anos_exp_seg_map)

df['especialista'] = df[coluna_especialistas].apply(
    lambda x: 1 if x in resposta_especialistas else 0)

df['experiencia_direta'] = df[coluna_experiencia].apply(
    lambda x: 1 if x == tem_experiencia else 0)

df['certificacao'] = df[coluna_certificacao].apply(
    lambda x: 0 if x in resposta_nao_certificacao else 1)

df['Qual é o seu nível de educação formal concluída?'] = df['Qual é o seu nível de educação formal concluída?'].map(
    nivel_educacao_map)

df['interacao_exp_direta_especialista'] = df.apply(
    interacao_exp_direta_especialista, axis=1)

df['relacao_exp_direta_especialista'] = df.apply(
    relacao_exp_direta_especialista, axis=1)

In [43]:
print(df['Qual é o seu nível de educação formal concluída?'].unique())
print(df[coluna_senioridade].unique())
print(df[coluna_anos_experiencia_dev].unique())
print(df[coluna_anos_experiencia_seg].unique())

[5 6 2 7 3 1 4]
[4 2 3 1]
[4 3 2 1 5]
[4 0 2 3 1 5]


In [44]:
df['especialista'].sum()

np.int64(26)

In [45]:
print("Total:", df.shape[0])
print(
    f"Porcentagem de experiencia direta: {(df_exp_direta.shape[0] / df.shape[0])*100:.3f}%")
print(
    f"Porcentagem de experiencia indireta: {(df_exp_indireta.shape[0] / df.shape[0])*100:.3f}%")

Total: 83
Porcentagem de experiencia direta: 74.699%
Porcentagem de experiencia indireta: 25.301%


In [46]:
df['relacao_exp_direta_especialista'].value_counts()

relacao_exp_direta_especialista
0.5    46
1.0    21
0.0    16
Name: count, dtype: int64

In [47]:
df_frequencia = df[colunas_frequencia].copy()
df_facilidade = df[colunas_facilidade].copy()
df_efetividade = df[colunas_efetividade].copy()
df_essencialidade = df[colunas_essencialidade].copy()

In [48]:
nivel_educacao = df['Qual é o seu nível de educação formal concluída?']
anos_exp_dev = df['Quantos anos de experiência você tem em funções relacionadas ao desenvolvimento de software ou TI?']
anos_exp_seg = df['Quantos anos de experiência você tem em funções relacionadas a Segurança de Software?']
senioridade = df['Por favor, indique a senioridade da posição que ocupa.']
interacao_exp_direta_especialista_col = df['interacao_exp_direta_especialista']
relacao_exp_direta_especialista_col = df['relacao_exp_direta_especialista']
certificacao = df['certificacao']

# formula peso 1
# df['peso'] = (0.3*nivel_educacao)+(0.25*anos_exp_dev)+(0.2*senioridade)+(0.25*interacao_exp_direta_especialista_col)

# formula peso 2
# df_final['peso'] = (nivel_educacao/nivel_educacao.nunique()) + (anos_exp_dev/anos_exp_dev.median()) + (senioridade/senioridade.median()) + relacao_exp_direta_especialista_col
df['peso'] = (nivel_educacao/nivel_educacao.nunique()) + certificacao + (anos_exp_dev/anos_exp_dev.median()) + \
    (anos_exp_seg/anos_exp_seg.median()) + (senioridade /
                                            senioridade.median()) + relacao_exp_direta_especialista_col


total_values_facilidade = {}
total_values_frequencia = {}
total_values_efetividade = {}
total_values_essencialidade = {}

total_values_facilidade_normalized = {}
total_values_frequencia_normalized = {}
total_values_efetividade_normalized = {}
total_values_essencialidade_normalized = {}


for coluna in df.columns:
    df_sem_na_col = df[df[coluna] != 0]  # remover os not applicable
    if coluna in colunas_frequencia:
        total_values_frequencia[coluna] = (
            df_sem_na_col[coluna] * df_sem_na_col['peso']).sum()
        total_values_frequencia_normalized[coluna] = (
            df_sem_na_col[coluna] * df_sem_na_col['peso']).sum() / (df_sem_na_col['peso'].sum()*4)

    if coluna in colunas_facilidade:
        total_values_facilidade[coluna] = (
            df_sem_na_col[coluna] * df_sem_na_col['peso']).sum()
        total_values_facilidade_normalized[coluna] = (
            df_sem_na_col[coluna] * df_sem_na_col['peso']).sum() / (df_sem_na_col['peso'].sum()*5)

    if coluna in colunas_efetividade:
        total_values_efetividade[coluna] = (
            df_sem_na_col[coluna] * df_sem_na_col['peso']).sum()
        total_values_efetividade_normalized[coluna] = (
            df_sem_na_col[coluna] * df_sem_na_col['peso']).sum() / (df_sem_na_col['peso'].sum()*5)

    if coluna in colunas_essencialidade:
        total_values_essencialidade[coluna] = (
            df_sem_na_col[coluna] * df_sem_na_col['peso']).sum()
        total_values_essencialidade_normalized[coluna] = (
            df_sem_na_col[coluna] * df_sem_na_col['peso']).sum() / (df_sem_na_col['peso'].sum()*5)

for coluna in df.columns:
    if coluna in colunas_facilidade + colunas_efetividade + colunas_frequencia + colunas_essencialidade:
        df[coluna] = df[coluna] * df['peso']

In [49]:
medias = []

for col in df_efetividade.columns:
    medias.append(df_efetividade.loc[df_efetividade[col].isin(
        [1, 2, 3, 4, 5]).values, col].mean())

## Criar a planilha

In [50]:
import re

# Definição das respostas diretas, indiretas, grandes e pequenas
resposta_direta = 1
resposta_indireta = 0

resposta_pequena = ["Até 9 funcionários",
                    "De 10 a 49 funcionários", "De 50 a 99 funcionários"]
resposta_grande = ["De 100 a 499 funcionários",
                   "De 500 a 999 funcionários", "Mais de 1000 funcionários"]

resposta_especialista = 1
resposta_nao_especialista = 0

# Nome das colunas corretas
coluna_experiencia = 'experiencia_direta'
coluna_tamanhoempresa = 'Qual o tamanho da empresa para a qual você trabalha atualmente?'
coluna_especialistas = 'especialista'
coluna_sistemas = 'Quais são os tipos de sistemas que você trabalha?'

# Filtra as respostas diretas e indiretas
diretas = df[df[coluna_experiencia] == resposta_direta].copy()
indiretas = df[df[coluna_experiencia] == resposta_indireta].copy()

# Filtra as respostas de empresas grandes e pequenas
grandes = df[df[coluna_tamanhoempresa].isin(resposta_grande)].copy()
pequenas = df[df[coluna_tamanhoempresa].isin(resposta_pequena)].copy()

especialistas = df[df[coluna_especialistas] == resposta_especialista].copy()
nao_especialistas = df[df[coluna_especialistas]
                       == resposta_nao_especialista].copy()

In [51]:
print("Quantidade de participantes no grupo Experiência Direta:", len(diretas))
print("Quantidade de participantes no grupo Experiência Indireta:", len(indiretas))
print("--"*35)

print("Quantidade de participantes no grupo Big:", len(grandes))
print("Quantidade de participantes no grupo Small:", len(pequenas))
print("--"*35)

print("Quantidade de participantes no grupo Especialistas:", len(especialistas))
print("Quantidade de participantes no grupo Não Especialistas:",
      len(nao_especialistas))
print("--"*35)
print(f"Porcentagem de participantes no grupo Big: {len(grandes)/88:.1f}")
print(f"Porcentagem de participantes no grupo Small: {len(pequenas)/88:.1f}")

Quantidade de participantes no grupo Experiência Direta: 62
Quantidade de participantes no grupo Experiência Indireta: 21
----------------------------------------------------------------------
Quantidade de participantes no grupo Big: 65
Quantidade de participantes no grupo Small: 16
----------------------------------------------------------------------
Quantidade de participantes no grupo Especialistas: 26
Quantidade de participantes no grupo Não Especialistas: 57
----------------------------------------------------------------------
Porcentagem de participantes no grupo Big: 0.7
Porcentagem de participantes no grupo Small: 0.2


In [52]:
from scipy import stats
from cliffs_delta import cliffs_delta

resultados = []

# Função que calcula as médias, p-value, effect size e retorna os resultados para um par de grupos


def calcular_dados(grupo1, grupo2, coluna):

    grupo1 = grupo1[coluna]
    grupo2 = grupo2[coluna]
    score_grupo1 = grupo1.mean()
    score_grupo2 = grupo2.mean()

    stat, p_value = stats.mannwhitneyu(
        grupo1.dropna(), grupo2.dropna(), alternative='greater')

    # cliffs delta
    effect_size = cliffs_delta(grupo1, grupo2)

    return score_grupo1, score_grupo2, p_value, effect_size

# Função que calcula a distribuição das respostas na escala Likert e a média ponderada para cada coluna


def calcular_likert_distribuicao(grupo, coluna):

    grupo_mapeadas = grupo[coluna]
    total_respostas = grupo_mapeadas.value_counts().sort_index()
    media_ponderada = sum(total_respostas.index *
                          (total_respostas / total_respostas.sum()))
    return total_respostas.to_dict(), media_ponderada

In [53]:
def detectar_pratica_e_atributo(texto: str):
    """
    Detecta o trecho entre colchetes e identifica se contém as substrings
    'facilidade' ou 'efetividade'.

    Retorna um dicionário com:
        - 'pratica': valor entre colchetes (ou None se não existir)
        - 'atributo': 'facilidade', 'efetividade' ou None
    """
    # --- 1. Detectar prática (trecho entre colchetes) ---
    match = re.search(r'\[(.*?)\]', texto)
    pratica = match.group(1).strip() if match else None

    # --- 2. Detectar atributo ---
    atributo = None
    if 'facilidade' in texto.lower():
        atributo = 'facilidade'
    elif 'efetividade' in texto.lower():
        atributo = 'efetividade'
    elif 'frequência' in texto.lower():
        atributo = 'frequência'
    elif 'essencial' in texto.lower():
        atributo = 'essencialidade'

    return pratica, atributo

In [54]:
# Loop que percorre cada coluna de preocupação, realiza as análises e armazena os resultados
for coluna in colunas_preocupacao:
    pratica, atributo = detectar_pratica_e_atributo(coluna)

    total_respostas, media_ponderada = calcular_likert_distribuicao(df, coluna)

    score_direto, score_indireto, p_value_direto_indireto, effect_size_direto_indireto = calcular_dados(
        diretas, indiretas, coluna)
    score_grande, score_pequena, p_value_grande_pequena, effect_size_grande_pequena = calcular_dados(
        grandes, pequenas, coluna)
    score_especialistas, score_nao_especialistas, p_value_especialistas_nao_especialistas, effect_size_especialistas_nao_especialistas = calcular_dados(
        especialistas, nao_especialistas, coluna)

    # Cria um dicionário com todos os resultados calculados e o adiciona à lista de resultados
    linha = {
        'Prática': pratica,
        'Atributo': atributo,
        'Likert Média': media_ponderada,
        'Média indireto': score_indireto,
        'Média direto': score_direto,
        'P-value Indirect/Direct': p_value_direto_indireto,
        'Effect Size Indirect/Direct': effect_size_direto_indireto,

        'Média Grande': score_grande,
        'Média Pequena': score_pequena,
        'P-value Big/Small': p_value_grande_pequena,
        'Effect Size Specialist/Non-Specialist': effect_size_grande_pequena,

        'Média Especialista': score_especialistas,
        'Média Não-Especialista': score_nao_especialistas,
        'P-value Specialist/Non-Specialist': p_value_especialistas_nao_especialistas,
        'Effect Size Especialista/Não-Especialista': effect_size_especialistas_nao_especialistas,
    }
    resultados.append(linha)

In [55]:
resultados

[{'Prática': 'Revisão de Código',
  'Atributo': 'facilidade',
  'Likert Média': 12.963568559954101,
  'Média indireto': np.float64(6.846938775510203),
  'Média direto': np.float64(15.035330261136712),
  'P-value Indirect/Direct': np.float64(3.6027090374188485e-05),
  'Effect Size Indirect/Direct': (0.5821812596006144, 'large'),
  'Média Grande': np.float64(12.991941391941394),
  'Média Pequena': np.float64(13.370535714285712),
  'P-value Big/Small': np.float64(0.591578852630622),
  'Effect Size Specialist/Non-Specialist': (-0.03653846153846154,
   'negligible'),
  'Média Especialista': np.float64(11.504578754578755),
  'Média Não-Especialista': np.float64(13.629072681704262),
  'P-value Specialist/Non-Specialist': np.float64(0.8335340976221617),
  'Effect Size Especialista/Não-Especialista': (-0.13225371120107962,
   'negligible')},
 {'Prática': 'Modelagem de Ameaças',
  'Atributo': 'facilidade',
  'Likert Média': 11.067412507171543,
  'Média indireto': np.float64(5.051020408163266),
 

In [57]:
df = pd.DataFrame(resultados)
df.to_excel('resultado_nao_formatado.xlsx', index=False)

## Formatação da tabela

In [60]:
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
import pandas as pd
import os

# --- Caminhos ---
input_path = "resultado_nao_formatado.xlsx"
output_path = "analise_formatada.xlsx"

# --- Carrega o arquivo base ---
# Assumindo que 'analise_temp.xlsx' contém os dados de S1, S2, S3, etc.
# e as colunas necessárias para preencher as médias e p-values.
df = pd.read_excel(input_path)

# Verificação e mapeamento de colunas (ajuste conforme seu DF real)
# É crucial que o DataFrame 'df' tenha as colunas que você está tentando extrair:
# Ex: 'Likert Média', 'Direta vs. Indireta Média', 'p-value Direta vs. Indireta', etc.
colunas_map = {
    'Média Likert': 'Likert Média',
    'Média Direta': 'Direta vs. Indireta Média Direta', # Nome da coluna no seu df original
    'Média Indireta': 'Direta vs. Indireta Média Indireta', # Nome da coluna no seu df original
    'p-value D/I': 'Direta vs. Indireta p-value',
    'd D/I': 'Direta vs. Indireta d',
    # Adicione os outros mapeamentos aqui
}

headers = ['Prática', 'Atributo', 'Likert Média', 'Média indireto', 'Média direto', 'P-value Indirect/Direct', 'Effect Size Indirect/Direct', 'Média Grande', 'Média Pequena', 'P-value Big/Small', 'Effect Size Specialist/Non-Specialist', 'Média Especialista', 'Média Não-Especialista', 'P-value Specialist/Non-Specialist', 'Effect Size Especialista/Não-Especialista']


print("Colunas detectadas no arquivo de entrada:", list(df.columns))

# --- Cria planilha nova *sem* cabeçalho inicial do Pandas ---
# Isso impede que o Pandas crie automaticamente uma linha 1 indesejada.
writer = pd.ExcelWriter(output_path, engine='openpyxl')
# Crie uma planilha vazia, se não houver dados ainda, ou use openpyxl diretamente.
# Alternativamente, crie um DataFrame temporário com 0 linhas para iniciar.
pd.DataFrame().to_excel(writer, sheet_name="Análise", index=False)
writer.close()

# --- Carrega a planilha para formatação com Openpyxl ---
wb = openpyxl.load_workbook(output_path)
ws = wb["Análise"]

# --- Definição dos Cabeçalhos ---
# Linha 1: Cabeçalhos Principais (que serão mesclados)
main_headers = [
    'Afirmação', 'ID', 'Likert',
    'Direta vs. Indireta',
    'Grande vs. Pequena',
    'Especialista vs. Não especialista'
]

# Linha 2: Subcolunas (Média, p-value, d)
# Note que 'Afirmação' e 'ID' não têm subcolunas, então suas células na linha 2 serão mescladas na linha 1.
sub_headers = [
    'Média', # Coluna 3
    'Média', 'Média', 'p-value', 'd', # Colunas 4 a 7
    'Média', 'Média', 'p-value', 'd', # Colunas 8 a 11
    'Média', 'Média', 'p-value', 'd'  # Colunas 12 a 15
]

# Mapeamento para Mesclagem (Colunas em que o cabeçalho principal se estende por subcolunas)
# Formato: (start_col, end_col, row)
merge_cells = [
    # Colunas Afirmação e ID (Mescladas verticalmente - Linhas 1 e 2)
    (1, 1, 1, 2), # Afirmação
    (2, 2, 1, 2), # ID
    # Coluna Likert (Mesclada verticalmente - Linhas 1 e 2)
    (3, 3, 1, 2), # Likert Média
    # Direta vs. Indireta (Mesclada horizontalmente - Colunas 4 a 7 na Linha 1)
    (4, 7, 1, 1),
    # Grande vs. Pequena (Mesclada horizontalmente - Colunas 8 a 11 na Linha 1)
    (8, 11, 1, 1),
    # Especialista vs. Não especialista (Mesclada horizontalmente - Colunas 12 a 15 na Linha 1)
    (12, 15, 1, 1)
]

# --- Estilos ---
header_font = Font(bold=True)
header_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid") # Um cinza/azul claro
center_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
thin_border = Border(
    left=Side(style="thin"),
    right=Side(style="thin"),
    top=Side(style="thin"),
    bottom=Side(style="thin")
)

# --- Montar os cabeçalhos nas linhas 1 e 2 ---

# 1. Aplicar Mesclagens
for start_col, end_col, start_row, end_row in merge_cells:
    ws.merge_cells(start_row=start_row, start_column=start_col, end_row=end_row, end_column=end_col)

# 2. Inserir os textos da Linha 1
# Índice para 'main_headers'. O start_col deve corresponder ao índice + 1.
col_index = 1
for header in main_headers:
    cell = ws.cell(row=1, column=col_index, value=header)
    cell.alignment = center_align
    cell.font = header_font
    cell.fill = header_fill
    # Avança o índice da coluna de acordo com a largura da coluna principal
    if header == 'Likert':
        col_index += 1 # Likert ocupa 1 coluna
    elif header in ('Direta vs. Indireta', 'Grande vs. Pequena', 'Especialista vs. Não especialista'):
        col_index += 4 # Estes ocupam 4 subcolunas
    else:
        col_index += 1 # Afirmação e ID ocupam 1 coluna

# 3. Inserir os textos da Linha 2 (Subcolunas)
# Começa da coluna 4, onde a primeira subcoluna de 'Direta vs. Indireta' inicia.
col_index = 4 # Primeira subcoluna real (Média Direta)
# A subcoluna 'Média' da Likert vai na coluna 3, mas a célula na linha 1 já está mesclada com a linha 2.
# Vamos focar nas subcolunas que precisam de texto na Linha 2: Colunas 4 a 15.
# Os nomes das subcolunas para os grupos Direta/Grande/Especialista (4 colunas cada)
group_sub_headers = ['Direta Média', 'Indireta Média', 'p-value', 'd']
full_sub_headers = group_sub_headers * 3 # 3 grupos de 4

current_col = 4
for i, sub_header in enumerate(full_sub_headers):
    cell = ws.cell(row=2, column=current_col, value=sub_header)
    cell.alignment = center_align
    cell.font = Font(bold=True) # Não tão forte quanto o principal, talvez.
    cell.fill = header_fill
    current_col += 1

# 4. Aplicar Borda e Estilo em todas as células de cabeçalho (Linhas 1 e 2)
for row in ws.iter_rows(min_row=1, max_row=2, min_col=1, max_col=15):
    for cell in row:
        cell.border = thin_border
        cell.alignment = center_align


# --- Escrita do corpo ---
current_row = 3  # Dados começam na linha 3 (após 2 linhas de cabeçalho)

# Mapeamento dos headers de destino para as colunas do seu DataFrame de entrada
# ESTE MAPEAMENTO PRECISA SER REVISADO COM OS NOMES EXATOS DAS COLUNAS DO SEU ARQUIVO DE ENTRADA (df)
data_col_map = {
    # Coluna na Planilha | Coluna no seu DF de Entrada
    3: 'Likert Média', # Coluna 3 - Likert Média
    4: 'Média Direta', # Coluna 4 - Direta Média
    5: 'Média Indireta', # Coluna 5 - Indireta Média
    6: 'p-value Direta vs. Indireta', # Coluna 6
    7: 'd Direta vs. Indireta', # Coluna 7
    8: 'Média Grande', # Coluna 8 - Grande Média
    9: 'Média Pequena', # Coluna 9 - Pequena Média
    10: 'p-value Grande vs. Pequena', # Coluna 10
    11: 'd Grande vs. Pequena', # Coluna 11
    12: 'Média Especialista', # Coluna 12 - Especialista Média
    13: 'Média Não Especialista', # Coluna 13 - Não Especialista Média
    14: 'p-value Especialista vs. Não Especialista', # Coluna 14
    15: 'd Especialista vs. Não Especialista', # Coluna 15
    16: 'ID', # ID (Será inserido na Coluna 2)
}

# Assumindo que seu DF de entrada tem uma coluna chamada 'Prática' e outra 'Atributo' (ou S1, S2, etc)
# Agrupamento por prática
for pratica, grupo in df.groupby("Prática"):
    num_linhas = len(grupo)
    start_row = current_row
    end_row = current_row + num_linhas - 1

    # --- Mesclar célula da prática (verticalmente) na Coluna 1 ---
    ws.merge_cells(start_row=start_row, start_column=1, end_row=end_row, end_column=1)
    pratica_cell = ws.cell(row=start_row, column=1, value=pratica)
    pratica_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    pratica_cell.font = Font(bold=True)
    pratica_cell.border = thin_border

    # --- Linhas da prática ---
    for i, (_, row) in enumerate(grupo.iterrows()):
        row_index = start_row + i
        for col, header in enumerate(headers[1:], start=2):  # pula a coluna 'Prática'
            value = row.get(header)

            # Tratar valores NaN ou None que causam células em branco e formatar outros tipos
            if value is None or (isinstance(value, float) and pd.isna(value)):
                display_value = "-"  # Usar um hífen para dados ausentes
            # Formatar a tupla de effect size, ex: (0.5, 'large') -> "0.500 (large)"
            elif isinstance(value, tuple) and len(value) == 2:
                try:
                    display_value = f"{value[0]:.3f} ({value[1]})"
                except (TypeError, IndexError):
                    display_value = str(value)
            # Formatar floats com 3 casas decimais
            elif isinstance(value, float):
                display_value = f"{value:.3f}"
            else:
                display_value = value

            cell = ws.cell(row=row_index, column=col, value=display_value)
            cell.alignment = center_align
            cell.border = thin_border

    current_row = end_row + 1

# --- Ajuste automático de largura ---
for col in range(1, 16): # 15 colunas no total
    col_letter = get_column_letter(col)
    width = 10 if col > 2 else 20
    ws.column_dimensions[col_letter].width = width

# --- Salvar ---
wb.save(output_path)
print(f"Arquivo formatado com cabeçalho de duas linhas e salvo em: {output_path}")

Colunas detectadas no arquivo de entrada: ['Prática', 'Atributo', 'Likert Média', 'Média indireto', 'Média direto', 'P-value Indirect/Direct', 'Effect Size Indirect/Direct', 'Média Grande', 'Média Pequena', 'P-value Big/Small', 'Effect Size Specialist/Non-Specialist', 'Média Especialista', 'Média Não-Especialista', 'P-value Specialist/Non-Specialist', 'Effect Size Especialista/Não-Especialista']
Arquivo formatado com cabeçalho de duas linhas e salvo em: analise_formatada.xlsx
